# Practice Lab: Handling Production Incidents (Lesson 12.9)

Module 12 . 4 exercises . Playbooks + scenario walkthrough + auto-rollback + chaos monkey


# Lesson 12.9: Handling Production Incidents

4 exercises: 2E + 1M + 1C

All exercises run **locally** (pure Python simulations).


In [ ]:
import random
from datetime import datetime
random.seed(42)


---
## Exercise 1 (Easy): Write Your Playbook


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Playbook:
    REQUIRED=["name","severity","detection","steps","rollback","escalation"]
    SEVS=["P1-Critical","P2-High","P3-Medium","P4-Low"]
    def __init__(self,**kw): self.f=kw
    def validate(self):
        miss=[f for f in self.REQUIRED if f not in self.f]
        issues=[]
        if miss: issues.append(f"Missing: {miss}")
        if self.f.get("severity") not in self.SEVS: issues.append(f"Bad severity")
        if len(self.f.get("steps",[]))<3: issues.append("Too few steps")
        return len(issues)==0,issues
    def show(self):
        print(f"  [{self.f['severity']}] {self.f['name']}")
        print(f"  Detection: {self.f['detection']}")
        for s in self.f.get("steps",[]): print(f"    {s}")
        print(f"  Rollback: {self.f['rollback']}")
        print(f"  Escalation: {self.f['escalation']}")

pb1=Playbook(name="Rate Limit Exceeded",severity="P2-High",
    detection="429 errors > 30% for 2+ min",
    steps=["1. Check which provider rate limiting","2. Check per-team usage",
        "3. Throttle top consumer","4. Enable queue backpressure",
        "5. Contact provider for quota increase","6. Redistribute to fallback providers"],
    rollback="Enable queue rate limiting. Throttle top consumer.",
    escalation="If >50% failing: page on-call + inform teams")

pb2=Playbook(name="Silent Model Version Change",severity="P2-High",
    detection="Eval score drops >15% with no code changes",
    steps=["1. Check provider changelog","2. Run prompt regression suite",
        "3. Compare new vs baseline on 50 cases","4. Pin to previous model version",
        "5. Update baseline if improved","6. Add model version to logs"],
    rollback="Pin model to previous version in config.",
    escalation="If quality >25% drop: page product lead")

print("Playbooks:")
for i,pb in enumerate([pb1,pb2],1):
    ok,issues=pb.validate()
    print(f"\n  Playbook {i}: {'VALID' if ok else 'INVALID'}")
    if issues: [print(f"    {x}") for x in issues]
    pb.show()


</details>


---
## Exercise 2 (Easy): Scenario Walkthrough


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from datetime import datetime

class IR:
    def __init__(self,scenario,sev): self.sc=scenario; self.sev=sev; self.tl=[]; self.acts=[]
    def log(self,t,event,phase): self.tl.append({"t":t,"e":event,"p":phase})
    def act(self,desc,cmd=""): self.acts.append({"d":desc,"c":cmd})

ir=IR("Team Beta agent: 15K calls in 2hrs ($180 vs $40 normal)","P2-High")
ir.log("23:00","Cost alert: $180 (4.5x normal)","DETECT")
ir.log("23:01","On-call acknowledges","DETECT")
ir.log("23:02","Dashboard: Team Beta 15K calls","TRIAGE")
ir.log("23:04","Agent /v1/agent in infinite loop","TRIAGE")
ir.log("23:06","Disable Team Beta API key","MITIGATE"); ir.act("Disable key","disable key-team-beta")
ir.log("23:07","Confirm: no more Beta calls","MITIGATE"); ir.act("Budget hard-cap","set cap=true")
ir.log("23:15","Root cause: max_steps=unlimited","RESOLVE"); ir.act("Fix max_steps=25","config update")
ir.log("23:20","Deploy fix, re-enable key with cap","RESOLVE"); ir.act("Re-enable","enable key, cap=$50")
ir.log("23:30","Incident resolved. Post-mortem drafted.","POST-MORTEM")

print("Scenario Walkthrough: Cost Explosion")
print(f"  Severity: {ir.sev}")
print(f"\n  Timeline:")
for t in ir.tl: print(f"    [{t['p']:<10}] {t['t']} {t['e']}")
print(f"\n  Actions: {len(ir.acts)}")
for a in ir.acts: print(f"    {a['d']}" + (f" -> {a['c']}" if a['c'] else ""))
print(f"\n  Post-Mortem:")
print(f"    Root cause: agent max_steps=unlimited, infinite re-ask loop")
for ai in [("Set max_steps=25 default","Sarthak","Tomorrow"),("Add per-team budget caps","Priya","This week"),
    ("Alert on loop count >50","Raj","This week"),("Agent cost dashboard","Amit","Next sprint")]:
    print(f"    [{ai[2]:<12}] {ai[0]} ({ai[1]})")


</details>


---
## Exercise 3 (Medium): Automated Rollback


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class AutoRB:
    def __init__(self,th=20,sustain=3,interval=30):
        self.th=th; self.sus=sustain; self.intv=interval; self.breach=0; self.rolled=False; self.log=[]
    def check(self,tick,total,errors):
        rate=errors/max(total,1)*100; hit=rate>self.th
        if hit: self.breach+=1
        else: self.breach=0
        status="OK"
        if hit: status=f"BREACH ({self.breach}/{self.sus})"
        if self.breach>=self.sus and not self.rolled:
            self.rolled=True; status="AUTO-ROLLBACK"
            self.log.append({"tick":tick,"cmd":"gcloud run services update-traffic app --to-revisions PREV=100"})
        self.log.append({"tick":tick,"rate":round(rate,1),"status":status})
        return status

m=AutoRB(20,3,30)
print("Automated Rollback Monitor:")
print(f"  {'Tick':>4} {'Req':>4} {'Err':>4} {'Rate':>6} {'Status':>18}")
rb_tick=first_b=None
for tick in range(1,21):
    if tick<=8: total=random.randint(80,120); errors=random.randint(1,5)
    elif tick<=15 and not m.rolled: total=random.randint(80,120); errors=random.randint(25,45)
    else: total=random.randint(80,120); errors=random.randint(1,5)
    st=m.check(tick,total,errors)
    print(f"  {tick:>4} {total:>4} {errors:>4} {errors/total*100:>5.1f}% {st:>18}")
    if "BREACH" in st and first_b is None: first_b=tick
    if st=="AUTO-ROLLBACK" and rb_tick is None: rb_tick=tick

if first_b and rb_tick:
    dt=(rb_tick-first_b)*m.intv
    print(f"\n  First breach: tick {first_b} | Rollback: tick {rb_tick}")
    print(f"  Detection to rollback: {dt}s ({dt//60}m{dt%60}s)")
for e in m.log:
    if "cmd" in e: print(f"  Command: {e['cmd']}")


</details>


---
## Exercise 4 (Challenge): Chaos Monkey


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

INCIDENTS={"provider_down":{"err":60,"lat":5000,"cost":1,"pii":False},
    "cost_spike":{"err":2,"lat":800,"cost":10,"pii":False},
    "pii_leak":{"err":1,"lat":800,"cost":1,"pii":True},
    "hallucination":{"err":5,"lat":800,"cost":1.5,"pii":False},
    "rate_limit":{"err":40,"lat":100,"cost":0.5,"pii":False}}

PLAYBOOKS={"provider_down":"Fallback chain + circuit breaker",
    "cost_spike":"Disable key + budget cap","pii_leak":"Block endpoint + purge cache + notify privacy",
    "hallucination":"Pin model version + regression tests","rate_limit":"Queue + throttle + quota request"}

def detect(m):
    if m["pii"]: return "pii_leak"
    if m["err"]>50: return "provider_down"
    if m["err"]>30 and m["lat"]<200: return "rate_limit"
    if m["cost"]>5: return "cost_spike"
    if m["err"]>3: return "hallucination"
    return "normal"

print("Chaos Monkey Simulation (50 steps):")
print(f"  {'Step':>4} {'Injected':>15} {'Detected':>15} {'Match':>6}")
inj=det=correct=fp=0
for step in range(1,51):
    if random.random()<0.20:
        actual=random.choice(list(INCIDENTS.keys())); m=INCIDENTS[actual]; inj+=1
        found=detect(m); det+=1; match=found==actual; correct+=match
        print(f"  {step:>4} {actual:>15} {found:>15} {'Y' if match else 'N':>6}  {PLAYBOOKS.get(found,'')[:30]}")
    else:
        nm={"err":random.uniform(0.5,3),"lat":800,"cost":1,"pii":False}
        if detect(nm)!="normal": fp+=1

acc=correct/max(inj,1)*100
print(f"\n  Injected: {inj} | Correct: {correct}/{inj} ({acc:.0f}%) | FP: {fp}")
print(f"  Target: >90% accuracy | Run monthly")


</details>
